In [36]:
## Neural Network Implementation (Before Optimization)

using Random
using Statistics
using LinearAlgebra
using Printf


# Layer_dense (Dense Layer) 

In [37]:
# Set a seed for reproducibility of random weights
Random.seed!(42)

# --- Layer_dense (Dense Layer) ---
mutable struct Layer_dense
    inputs::Matrix{Float32}
    weights::Matrix{Float32} 
    biases ::Matrix{Float32}
    output::Matrix{Float32} 
    dweights::Matrix{Float32}
    dbiases::Matrix{Float32}
    dinputs::Matrix{Float32}

    function Layer_dense(n_inputs::Int, n_neurons::Int)
        # Initialize weights with small random numbers from a Gaussian distribution
        weights = Float32(0.01) * randn(Float32, n_inputs, n_neurons)
        biases = zeros(Float32, 1, n_neurons)
        new(Matrix{Float32}(undef,0,0), weights, biases, Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0))
    end
end

# Forward Pass for dense layer
function forward(layer::Layer_dense, inputs::Matrix{Float32})
    # Store inputs for use in the backward pass
    layer.inputs = inputs 
    # Calculate output: Z = XW + B
    layer.output = inputs * layer.weights .+ layer.biases
end

# Backward Pass for dense layer
function backward(layer::Layer_dense, upstream_gradient::Matrix{Float32})
    # Gradient of the loss with respect to weights: dL/dW = X^T * dL/dZ
    layer.dweights = layer.inputs' * upstream_gradient
    # Gradient of the loss with respect to biases: dL/dB = sum(dL/dZ)
    layer.dbiases = sum(upstream_gradient, dims=1)
    # Gradient of the loss with respect to inputs: dL/dX = dL/dZ * W^T
    layer.dinputs = upstream_gradient * layer.weights'
end

backward (generic function with 3 methods)

# Activation_ReLU (ReLU Activation Function)

In [38]:
# --- Activation_ReLU (ReLU Activation Function) ---
mutable struct Activation_ReLU
    inputs::Matrix{Float32}
    output::Matrix{Float32}
    dinputs::Matrix{Float32}
    Activation_ReLU() = new(Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0))
end

# Forward pass for ReLU
function forward(activation::Activation_ReLU, inputs::Matrix{Float32})
    # Store inputs for the backward pass
    activation.inputs = inputs
    # Apply ReLU: max(0, input)
    activation.output = max.(Float32(0.0), inputs)
end

# Backward pass for ReLU
function backward(activation::Activation_ReLU, upstream_gradient::Matrix{Float32})
    # Start with a copy of the upstream gradient
    activation.dinputs = copy(upstream_gradient)
    # Zero out gradients where the original input was non-positive
    activation.dinputs[activation.inputs .<= 0] .= Float32(0.0)
end

backward (generic function with 3 methods)

# Activation_Softmax_Loss_CategoricalCrossentropy

In [39]:
mutable struct Activation_Softmax_Loss_CategoricalCrossentropy
    output::Matrix{Float32} # This will store the softmax probabilities
    dinputs::Matrix{Float32}
    
    Activation_Softmax_Loss_CategoricalCrossentropy() = new(Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0))
end

function forward(combo::Activation_Softmax_Loss_CategoricalCrossentropy, inputs::Matrix{Float32}, y_true::Vector{Int})
    # 1. Softmax Activation
    exp_values = exp.(inputs .- maximum(inputs, dims=2))
    probabilities = exp_values ./ sum(exp_values, dims=2)
    combo.output = probabilities

    # 2. Categorical Cross-Entropy Loss Calculation
    n_samples = size(probabilities, 1)
    # Clip data to prevent division by 0
    probabilities_clipped = clamp.(probabilities, Float32(1e-7), Float32(1.0) - Float32(1e-7))
    
    # Get the probabilities corresponding to the true labels
    correct_confidences = [probabilities_clipped[i, y_true[i]] for i in 1:n_samples]
    
    # Calculate negative log likelihoods and return the average loss
    negative_log_likelihoods = -log.(correct_confidences)
    data_loss = mean(negative_log_likelihoods)
    return data_loss
end

function backward(combo::Activation_Softmax_Loss_CategoricalCrossentropy, y_true::Vector{Int})
    n_samples = size(combo.output, 1)
    n_outputs = size(combo.output, 2)

    # Copy probabilities
    combo.dinputs = copy(combo.output)

    # Calculate gradient
    # For each sample, subtract 1 from the probability of the true class
    for i in 1:n_samples
        combo.dinputs[i, y_true[i]] -= Float32(1.0)
    end

    # Normalize gradient by the number of samples
    combo.dinputs ./= n_samples
end

backward (generic function with 3 methods)

# Data Generation Function

In [40]:
function create_data(n_points::Int, n_classes::Int)
    X = zeros(Float32, n_points * n_classes, 2) # Features
    y = zeros(Int, n_points * n_classes)        # Labels

    for class_number in 0:(n_classes - 1)
        ix = (class_number * n_points + 1):((class_number + 1) * n_points)
        
        # Radius for the spiral
        r = range(Float32(0.0), Float32(1.0), length=n_points)
        
        # Angle for the spiral
        t = range(class_number * Float32(4.0), (class_number + 1) * Float32(4.0), length=n_points) .+ (randn(Float32, n_points) * Float32(0.2))
        
        # Calculate x and y coordinates
        X[ix, 1] = r .* sin.(t * Float32(2.5))
        X[ix, 2] = r .* cos.(t * Float32(2.5))
        
        # Assign class labels (Julia is 1-indexed, so add 1)
        y[ix] .= class_number + 1
    end
    return X, y
end

create_data (generic function with 1 method)

# Main Execution

In [41]:
# Generate the spiral dataset
X, y = create_data(100, 3)
println("--- Dataset Generated ---")
println("Shape of X: ", size(X))
println("Shape of y: ", size(y))
println("First 5 samples of X:\n", X[1:5,:])
println("First 5 labels of y:\n", y[1:5])

# Define the network architecture
layer1 = Layer_dense(size(X, 2), 64)
activation1 = Activation_ReLU()
layer2 = Layer_dense(64, size(unique(y), 1)) # Number of unique classes in y
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

println("\n--- Network Architecture Initialized ---")
println("Layer 1 Weights shape: ", size(layer1.weights))
println("Layer 1 Biases shape: ", size(layer1.biases))
println("Layer 2 Weights shape: ", size(layer2.weights))
println("Layer 2 Biases shape: ", size(layer2.biases))

# --- Training Loop (10 Epochs - No Optimization) ---
println("\n--- Starting Training Loop (10 Epochs - No Optimization) ---")
println("Note: Weights and biases will NOT be updated in this loop.")
println("      Loss and accuracy will remain constant as there's no optimizer.")

n_epochs = 10

for epoch in 1:n_epochs
    # --- Forward Pass ---
    forward(layer1, X)
    forward(activation1, layer1.output)
    forward(layer2, activation1.output)
    data_loss = forward(loss_activation, layer2.output, y)
    
    # Calculate accuracy
    predictions = [argmax(row) for row in eachrow(loss_activation.output)]
    accuracy = mean(predictions .== y)
    
    # --- Backward Pass ---
    # Backward pass for Softmax + Categorical Cross-entropy
    backward(loss_activation, y)
    
    # Backward pass for Layer 2 (Dense)
    backward(layer2, loss_activation.dinputs)
    
    # Backward pass for Activation 1 (ReLU)
    backward(activation1, layer2.dinputs)
    
    # Backward pass for Layer 1 (Dense)
    backward(layer1, activation1.dinputs)

    # --- Print Results for the Epoch ---
    @printf "Epoch %d: Loss = %.4f, Accuracy = %.4f\n" epoch data_loss accuracy
    
    # Optional: Print some gradients to verify they are being calculated
    # if epoch == 1 || epoch == n_epochs
    #     println("  Layer 1 dWeights (first 2x2): \n", layer1.dweights[1:min(2, size(layer1.dweights,1)), 1:min(2, size(layer1.dweights,2))])
    #     println("  Layer 2 dBiases: ", layer2.dbiases)
    # end
end


## Lets implement the OPTIMIZER_SGD


In [42]:
mutable struct Optimizer_SGD
    learning_rate::Float32
    function Optimizer_SGD(learning_rate::Float32=Float32(1.0))
        new(learning_rate)
    end
end
# Update parameters for a dense layer using SGD
function update_parameters(optimizer::Optimizer_SGD, layer::Layer_dense)
    layer.weights .+= -optimizer.learning_rate .* layer.dweights
    layer.biases .+= -optimizer.learning_rate .* layer.dbiases
end

update_parameters (generic function with 1 method)

In [ ]:
# Define the network architecture
layer1 = Layer_dense(size(X, 2), 64)
activation1 = Activation_ReLU()
layer2 = Layer_dense(64, size(unique(y), 1)) # Number of unique classes in y
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

# Define the optimizer
optimizer = Optimizer_SGD(Float32(0.5)) # Using a learning rate of 0.5

println("\n--- Network Architecture Initialized ---")
println("Layer 1 Weights shape: ", size(layer1.weights))
println("Layer 1 Biases shape: ", size(layer1.biases))
println("Layer 2 Weights shape: ", size(layer2.weights))
println("Layer 2 Biases shape: ", size(layer2.biases))
println("Optimizer Learning Rate: ", optimizer.learning_rate)


# --- Training Loop (With SGD Optimization) ---
println("\n--- Starting Training Loop (With SGD Optimization) ---")
println("Note: Weights and biases will now be updated, so loss and accuracy should change.")

n_epochs = 10000 # Changed to 10000 as per your previous request

# Initialize data_loss and accuracy outside the loop
data_loss::Float32 = Float32(0.0)
accuracy::Float32 = Float32(0.0)

for epoch in 1:n_epochs
    # --- Forward Pass ---
    forward(layer1, X)
    forward(activation1, layer1.output)
    forward(layer2, activation1.output)
    data_loss = forward(loss_activation, layer2.output, y)
    
    # Calculate accuracy
    predictions = [argmax(row) for row in eachrow(loss_activation.output)]
    accuracy = mean(predictions .== y)
    
    # --- Backward Pass ---
    backward(loss_activation, y)
    backward(layer2, loss_activation.dinputs)
    backward(activation1, layer2.dinputs)
    backward(layer1, activation1.dinputs)

    # --- Update Parameters (Optimization Step) ---
    update_parameters(optimizer, layer1)
    update_parameters(optimizer, layer2)

    # --- Print Results for the Epoch ---
    if epoch == 1 || epoch % 100 == 0 # Print for the first epoch and every 100th epoch
        @printf "Epoch %d: Loss = %.4f, Accuracy = %.4f\n" epoch data_loss accuracy
    end
end

println("\n--- Training Loop Complete ---")
println("Final Loss: ", data_loss)
println("Final Accuracy: ", accuracy)
println("\nNow you can see the loss decreasing and accuracy increasing over epochs!")

Activation_Softmax_Loss_CategoricalCrossentropy(Matrix{Float32}(undef, 0, 0), Matrix{Float32}(undef, 0, 0))